## Catalog Parser

Existing catalog parser function is analyzed first

In [2]:
from __future__ import annotations
import json
import logging

from cimgraph.databases import get_cim_profile
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj

Next, let us provide some inputs and see how it performs for these functions.

Two inputs are provided, a network, and a json file in particular structure.

In [3]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

Catalog_JSON_file_path = '../test_models/hv69_12.json'
Obj_output = catalog_parser(Catalog_JSON_file_path, network)

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Note that some attributes were noted as "not found"

Further, the output object does not have many attributes from the input.

In [4]:
print(Obj_output)

{"@id": "1cb3da12-0993-4a2f-a5f4-23f94c8f2b52", "@type": "PowerTransformer", "name": "hvmv69_12", "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "9f42dfbd-5ddd-4c14-aaeb-206aed79fac1", "@type": "PowerTransformerEnd"}, {"@id": "ffc77da4-9e0f-4551-be01-993bcea5105c", "@type": "PowerTransformerEnd"}]}


note that only two attributes for each winding - id and type - are reflected in the output for Object output. Where as, in the input many inputs like r,x,b, ratedS, ratedU among others for each windings things were available.

In [5]:
{
    "catalog": {
        "@type": "PowerTransformer",
        "name": "hvmv69_12",
        "vectorGroup": "Yy",
        "PowerTransformerEnd": [
            {
                "@type": "PowerTransformerEnd", 
                "name": "hvmv69_12_End_1",
                "endNumber": "1",
                "grounded": "true",
                "rground": "0",
                "xground": "0",
                "connectionKind": "WindingConnection.Y",
                "phaseAngleClock": "0",
                "r": "1.594935",
                "x": "18.90117",
                "b": "0",
                "ratedS": "20000000",
                "ratedU": "69000"
            },
            {
                "@type": "PowerTransformerEnd", 
                "name": "hvmv69_12_End_2",
                "endNumber": "2",
                "grounded": "true",
                "rground": "0",
                "xground": "0",
                "connectionKind": "WindingConnection.Y",
                "phaseAngleClock": "0",
                "r": "0.052092802",
                "x": "0",
                "b": "0",
                "ratedS": "20000000",
                "ratedU": "12470"
            }
        ]
    }
}

{'catalog': {'@type': 'PowerTransformer',
  'name': 'hvmv69_12',
  'vectorGroup': 'Yy',
  'PowerTransformerEnd': [{'@type': 'PowerTransformerEnd',
    'name': 'hvmv69_12_End_1',
    'endNumber': '1',
    'grounded': 'true',
    'rground': '0',
    'xground': '0',
    'connectionKind': 'WindingConnection.Y',
    'phaseAngleClock': '0',
    'r': '1.594935',
    'x': '18.90117',
    'b': '0',
    'ratedS': '20000000',
    'ratedU': '69000'},
   {'@type': 'PowerTransformerEnd',
    'name': 'hvmv69_12_End_2',
    'endNumber': '2',
    'grounded': 'true',
    'rground': '0',
    'xground': '0',
    'connectionKind': 'WindingConnection.Y',
    'phaseAngleClock': '0',
    'r': '0.052092802',
    'x': '0',
    'b': '0',
    'ratedS': '20000000',
    'ratedU': '12470'}]}}

This begs for a pertinent question. What is the ulterior objective for catalog parser? 

Is it providing all the relevant inputs in a more structure fashion? 

Is it stream lining inputs for Object Builder functions?

In light of these questions, let us try to modify the catalog parser function.

In [41]:
from __future__ import annotations
import json
import logging

from cimgraph.databases import get_cim_profile
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


def catalog_parser_2(catalog_file, network):
    print('Catalog parser started')
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile
    print('Data from catalog loaded') 
    obj = item_parser_2(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser_2(data:dict, network: GraphModel, cim):
    print('Item Parser started')
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data: ## 
        print('flag')
        print(attribute)
        if type(data[attribute]) == str:
            print('If string part active')
            if attribute in class_type.__dataclass_fields__: ## May be the attributes r, x, b, connection kind etc. where not passing this filter in the original file
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list: ## my interpretation - if string, the above function takes care of it, but if it is list, the elif is where it gets caught. It is a recursive fuction, as it calls item_parser again inside the elif criterion. Thus any kind of nested inputs can be handled.
            ##if attribute in class_type.__dataclass_fields__:
            print('Elif list part active')
            values = getattr(obj, attribute)
            print(data[attribute])
            for item in data[attribute]:
                print(item)
                value = item_parser_2(item, network, cim)
                values.append(value)
            setattr(obj, attribute, values)
    return obj

In [42]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

Catalog_JSON_file_path = '../test_models/hv69_12.json'
Obj_output_2 = catalog_parser_2(Catalog_JSON_file_path, network)

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Catalog parser started
Data from catalog loaded
Item Parser started
flag
@type
If string part active
flag
name
If string part active
flag
vectorGroup
If string part active
flag
PowerTransformerEnd
Elif list part active
[{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.90117', 'b': '0', 'ratedS': '20000000', 'ratedU': '69000'}, {'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_2', 'endNumber': '2', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '0.052092802', 'x': '0', 'b': '0', 'ratedS': '20000000', 'ratedU': '12470'}]
{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.9

In [57]:
print(Obj_output_2.PowerTransformerEnd[1])

{"@id": "a029ca38-6b1e-487c-8a39-67dfad2b6d8d", "@type": "PowerTransformerEnd", "name": "hvmv69_12_End_2", "endNumber": "2", "grounded": "true", "rground": "0", "xground": "0", "phaseAngleClock": "0", "connectionKind": "WindingConnection.Y", "r": "0.052092802", "ratedS": "20000000", "ratedU": "12470"}


In [52]:
from __future__ import annotations
import json
import logging

from cimgraph.databases import get_cim_profile
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


def catalog_parser_3(catalog_file, network):
    print('Catalog parser started')
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile
    print('Data from catalog loaded') 
    obj = item_parser_3(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser_3(data:dict, network: GraphModel, cim):
    print('Item Parser started')
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data: ## 
        print('flag')
        print(attribute)
        if type(data[attribute]) == str:
            print('If string part active')
            setattr(obj, attribute, data[attribute])
            # if attribute in class_type.__dataclass_fields__: ## May be the attributes r, x, b, connection kind etc. where not passing this filter in the original file
            #     setattr(obj, attribute, data[attribute])
            # else:
            #     _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list: ## my interpretation - if string, the above function takes care of it, but if it is list, the elif is where it gets caught. It is a recursive fuction, as it calls item_parser again inside the elif criterion. Thus any kind of nested inputs can be handled.
            ##if attribute in class_type.__dataclass_fields__:
            print('Elif list part active')
            values = getattr(obj, attribute)
            print(data[attribute])
            for item in data[attribute]:
                print(item)
                value = item_parser_3(item, network, cim)
                print(value)
                values.append(value)
            setattr(obj, attribute, values)
    return obj

In [53]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

Catalog_JSON_file_path = '../test_models/hv69_12.json'
Obj_output_3 = catalog_parser_3(Catalog_JSON_file_path, network)



Catalog parser started
Data from catalog loaded
Item Parser started
flag
@type
If string part active
flag
name
If string part active
flag
vectorGroup
If string part active
flag
PowerTransformerEnd
Elif list part active
[{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.90117', 'b': '0', 'ratedS': '20000000', 'ratedU': '69000'}, {'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_2', 'endNumber': '2', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '0.052092802', 'x': '0', 'b': '0', 'ratedS': '20000000', 'ratedU': '12470'}]
{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.9

In [54]:
print(Obj_output_3)

{"@id": "0069b282-370d-4a89-87ce-4b1ec3677304", "@type": "PowerTransformer", "name": "hvmv69_12", "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "ed2b3c61-07ce-48d1-8a52-fe4ff81e810e", "@type": "PowerTransformerEnd"}, {"@id": "229a65c8-7365-4c3c-9628-c17d7b9789f2", "@type": "PowerTransformerEnd"}]}
